<a href="https://colab.research.google.com/github/h-ardy/rosetta/blob/main/rosetta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Rosetta
Some small languages aren't translated well into English, or other underrepresented languages. That's where Rosetta comes in. It uses semantics to link common terms and connect languages that never would've been able to otherwise.

## What this notebook does

Courts, clinics and schools in the US regularly meet people who speak a language that no translation app covers and for which interpreters are hard to find. Translation tools skip these languages because there is too little translated text to train on.

Rosetta works from the little text that does exist. Given a few hundred translated sentences, it:

1. writes its own dictionary and grammar notes, where every dictionary entry points back to the examples that support it,
2. translates new sentences using only that dictionary and those notes, with a word-by-word gloss,
3. flags each translation with a confidence score computed in plain Python from the evidence counts, so a worker knows when to wait for a human interpreter,
4. attaches every learned word to a shared map of everyday concepts, so languages that have never shared a dictionary connect through meaning.

We test it on three languages from three regions: Southern Quechua (Andes), Shipibo-Konibo (Peruvian Amazon) and Pirahã (Brazilian Amazon). Results are reported separately for each language, on test sentences the system never saw.

**This is a prototype. It does not replace a human interpreter, and it should not be relied on in a medical, legal or other high-stakes setting.**

**How to run:** add a Gemini API key in Colab Secrets (key icon on the left) named `GEMINI_API_KEY`, then Runtime > Run all. Without a key, the notebook replays saved model responses from `cache/model_cache.json`, so every number below can be reproduced.

## 1. Setup

In [1]:
!pip -q install sacrebleu google-genai networkx

In [2]:
import socket; socket.setdefaulttimeout(60)
import os, re, json, time, random, hashlib, string, unicodedata, urllib.request
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import sacrebleu

pd.set_option("display.max_colwidth", 120)

# ---- Configuration: everything you might change lives here ----
SEED = 13
MAX_SRC_WORDS = 12        # keep short sentences; intake questions are short
N_TEST = 50               # held-out test sentences per language
POOL_SIZE = 200           # learning pool per language
SIZES_FULL = [25, 50, 100, 200]
SIZES_REDUCED = [50, 200]
CIPHER_SIZES = [50, 200]
CONFIDENCE_CUTOFF = 0.6   # below this, Rosetta says to wait for an interpreter
BATCH = 25                # test sentences per translation call (fewer calls = fewer rate limits)
MODEL = ""                # leave empty to pick the newest Gemini Flash model automatically
REPO_RAW = "https://raw.githubusercontent.com/h-ardy/rosetta/main/"
CACHE_PATH = "cache/model_cache.json"
os.makedirs("cache", exist_ok=True); os.makedirs("outputs", exist_ok=True); os.makedirs("data", exist_ok=True)

LANGS = {
    "quechua": dict(name="Southern Quechua", src_name="Spanish", region="Andes", plan="full"),
    "shipibo": dict(name="Shipibo-Konibo", src_name="Spanish", region="Peruvian Amazon", plan="reduced"),
    "piraha":  dict(name="Pirahã", src_name="English", region="Brazilian Amazon", plan="reduced"),
}

## 2. The cached model client

Every call to the language model goes through one function. It hashes the prompt, returns a saved answer if one exists, and otherwise calls Gemini and saves the answer. This makes the notebook reproducible without a key, and it means a crash or a rate limit never loses finished work.

The model only does two jobs in this project: writing the dictionary and translating. Every score is computed by plain Python further down.

In [8]:
def _download(url, path):
    try:
        urllib.request.urlretrieve(url, path); return True
    except Exception as e:
        print("Could not download", url, "->", e); return False

if not os.path.exists(CACHE_PATH):
    _download(REPO_RAW + CACHE_PATH, CACHE_PATH)
CACHE = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {"meta": {}, "responses": {}}

def save_cache():
    json.dump(CACHE, open(CACHE_PATH, "w"), ensure_ascii=False)

API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        API_KEY = None

client = None
if API_KEY:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=API_KEY)

def pick_model():
    # Newest stable Gemini Flash model that can generate text.
    names = []
    for m in client.models.list():
        n = m.name.split("/")[-1]
        acts = getattr(m, "supported_actions", None) or []
        if "flash" in n and "generateContent" in acts and not any(b in n for b in ["image", "tts", "audio", "live", "lite", "exp", "embedding", "thinking"]):
            names.append(n)
    def key(n):
        nums = [int(x) for x in re.findall(r"\d+", n)][:2]
        return ("preview" not in n, nums)
    if not names:
        names = [m.name.split("/")[-1] for m in client.models.list() if "flash" in m.name and not any(b in m.name for b in ["image", "tts", "audio", "live", "lite", "embedding"])]
    if not names:
        return "gemini-2.5-flash"
    names.sort(key=key)
    return names[-1]

if client and not MODEL:
    MODEL = CACHE["meta"].get("model") or pick_model()
elif not MODEL:
    MODEL = CACHE["meta"].get("model", "")
CACHE["meta"]["model"] = MODEL
print("Model:", MODEL or "(none)", "| API key:", "found" if API_KEY else "not found, replaying cache")
print("Cached responses:", len(CACHE["responses"]))

_last_call = [0.0]
CALLS = Counter()

def ask(prompt, tag="call"):
    # Send a prompt, return parsed JSON. Uses the cache first.
    k = hashlib.sha256((MODEL + "\n" + prompt).encode()).hexdigest()
    if k in CACHE["responses"]:
        CALLS["cached"] += 1
        return json.loads(CACHE["responses"][k])
    if client is None:
        raise RuntimeError("No API key and this prompt is not in the cache. Add GEMINI_API_KEY in Colab Secrets.")
    last_err = None
    for attempt in range(6):
        wait = 4 - (time.time() - _last_call[0])
        if wait > 0: time.sleep(wait)
        try:
            _last_call[0] = time.time()
            r = client.models.generate_content(
                model=MODEL, contents=prompt,
                config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"))
            text = r.text
            if not text:
                raise json.JSONDecodeError("empty response", "", 0)
            text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
            data = json.loads(text)
            CACHE["responses"][k] = text
            save_cache(); CALLS["live"] += 1
            return data
        except json.JSONDecodeError as e:
            last_err = e
            prompt = prompt + "\n\nYour last answer was not valid JSON. Return only valid JSON."
            k = hashlib.sha256((MODEL + "\n" + prompt).encode()).hexdigest()
        except Exception as e:
            last_err = e
            msg = str(e)
            if ("PerDay" in msg or "per day" in msg.lower()) and attempt >= 1:
                raise RuntimeError("Daily free-tier quota reached for " + MODEL + ". Progress is saved in the cache. "
                                   "Rerun later, or set MODEL to another Flash model listed in Google AI Studio and rerun.") from e
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg or "503" in msg or "UNAVAILABLE" in msg:
                pause = 20 * (attempt + 1)
                print(f"  rate limited on {tag}, waiting {pause}s")
                time.sleep(pause)
            else:
                raise
    raise RuntimeError(f"Gave up on {tag}: {last_err}")

Could not download https://raw.githubusercontent.com/h-ardy/rosetta/main/cache/model_cache.json -> HTTP Error 404: Not Found
Model: gemini-3.8-flash | API key: found
Cached responses: 0


## 3. Data

All three datasets are public and credited at the end.

* **Quechua and Shipibo-Konibo** come from the [AmericasNLP 2021 shared task](https://github.com/AmericasNLP/americasnlp2021). We use its development set: everyday sentences professionally translated from Spanish. Translation direction: Spanish into the Indigenous language, the direction the shared task uses and the direction an intake worker needs.
* **Pirahã** comes from the [MIT Pirahã Corpus](https://osf.io/kt2e8/) (Futrell et al. 2016, CC BY-SA), 1,149 sentences of natural speech recorded by Steve Sheldon and Daniel Everett, each with an English translation and word-by-word glosses. Direction: English into Pirahã. The linguists' glosses give us an answer key for the dictionary Rosetta writes.

For each language we keep sentences of at most 12 words, shuffle with a fixed seed, and set aside 50 test sentences **before** anything else happens. The model never sees a test sentence's translation.

In [6]:
ANLP = "https://raw.githubusercontent.com/AmericasNLP/americasnlp2021/main/data/"
PIRAHA_URL = "https://files.osf.io/v1/resources/kt2e8/providers/osfstorage/564d13858c5e4a31d66ffa69"

def fetch(url, path):
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (rosetta research notebook)"})
        try:
            data = urllib.request.urlopen(req, timeout=60).read()
        except Exception:
            req = urllib.request.Request(url + ("&" if "?" in url else "?") + "action=download", headers={"User-Agent": "Mozilla/5.0"})
            data = urllib.request.urlopen(req, timeout=60).read()
        open(path, "wb").write(data)
    return open(path, encoding="utf-8").read()

def load_americasnlp(folder, code):
    es = fetch(f"{ANLP}{folder}/dev.es", f"data/{code}.dev.es").splitlines()
    xx = fetch(f"{ANLP}{folder}/dev.{code}", f"data/{code}.dev.{code}").splitlines()
    return pd.DataFrame({"src": es, "tgt": xx}).assign(gloss=None)

LABEL = re.compile(r"\(\s*[A-Za-z][A-Za-z\-\?]*")
def parse_piraha(text):
    # Returns one row per sentence: English translation, Pirahã surface form, root glosses.
    rows = []
    blocks = re.split(r"\n(?=# \d+\.\d+\.\d+:?\s)", text)
    for b in blocks:
        m = re.match(r"# (\d+\.\d+\.\d+):?\s+(.*)", b)
        if not m: continue
        english = m.group(2).strip()
        english = re.sub(r"###.*", "", english)
        english = english.replace("[", "").replace("]", "").replace('"', "").strip()
        english = re.sub(r"\s*\(Lit:.*?\)\s*", " ", english).strip()
        parse = "\n".join(l for l in b.splitlines()[1:] if not l.startswith("#"))
        parse = LABEL.sub("(", parse)
        words, roots = [], []
        for tok in re.findall(r"[^\s()]+", parse):
            if tok == "*": continue
            parts = tok.split("/")
            form = parts[0]
            gloss = parts[1] if len(parts) > 1 else ""
            if form.startswith("-"):
                if words: words[-1] += form[1:]
            else:
                words.append(form)
                roots.append((form.lower(), gloss.replace("_", " ").lower()))
        if words and english:
            rows.append({"id": m.group(1), "src": english, "tgt": " ".join(words), "gloss": roots})
    return pd.DataFrame(rows)

def load_piraha():
    return parse_piraha(fetch(PIRAHA_URL, "data/piraha_corpus.txt"))

RAW = {
    "quechua": load_americasnlp("quechua-spanish", "quy"),
    "shipibo": load_americasnlp("shipibo_konibo-spanish", "shp"),
    "piraha":  load_piraha(),
}
for k, df in RAW.items():
    print(f"{LANGS[k]['name']:18s} {len(df):5d} sentence pairs")
RAW["piraha"].head(3)

Southern Quechua     996 sentence pairs
Shipibo-Konibo       996 sentence pairs
Pirahã              1149 sentence pairs


,id,src,tgt,gloss
0,1.1.1,Early in the day I spoke.,ti xahoagI ti iga OpIxi,"[(ti, 1), (xahoa, day), (ti, 1), (iga, speak), (o, vertical move)]"
1,1.1.2,BaIgipOhoasi speaker's sister spoke.,hi igA xai baIgipOhoasi,"[(hi, 3), (iga, speak), (xai, do), (baigipohoasi, baigipohoasi)]"
2,1.1.3,Is Kato sleepy? Lit: Kato-- her eyes flutter?,KatO hi obapIaagoxoihI KatO,"[(kato, kato), (hi, 3), (o, eye), (kato, kato)]"


In [7]:
def split(df, seed=SEED):
    df = df[df.src.str.split().str.len().between(2, MAX_SRC_WORDS) & (df.tgt.str.strip() != "")]
    df = df.drop_duplicates("src").sample(frac=1, random_state=seed).reset_index(drop=True)
    df["pid"] = range(len(df))
    test = df.iloc[:N_TEST].copy()
    pool = df.iloc[N_TEST:N_TEST + POOL_SIZE].copy()
    assert not set(test.src) & set(pool.src), "test sentence leaked into the pool"
    return pool, test

DATA = {k: split(df) for k, df in RAW.items()}
for k, (pool, test) in DATA.items():
    print(f"{LANGS[k]['name']:18s} pool {len(pool)}  test {len(test)}")
pd.DataFrame([{"language": LANGS[k]["name"], "source": test.src.iloc[0], "target": test.tgt.iloc[0]} for k, (pool, test) in DATA.items()])

Southern Quechua   pool 200  test 50
Shipibo-Konibo     pool 200  test 50
Pirahã             pool 200  test 50


,language,source,target
0,Southern Quechua,Fue un día relajante.,Allin samay punchawmi karqa.
1,Shipibo-Konibo,¿Por qué no nos explicas tu libro a todos?,¿Jawe kopiki jatibi min noa libroki yoiyamai?
2,Pirahã,"(I heard that as has been mentioned,) she Xiogi thus spoke.",hi aigIa ga xaisaihIai


## 4. Induction: the model writes a dictionary and grammar notes

The model sees N numbered sentence pairs from the learning pool and nothing else. It must return a dictionary where each entry lists the ids of the example pairs it came from. Python then **checks every claimed example**: an example only counts as evidence if the source word really appears in that pair's source sentence and the form really appears in its target sentence. The model cannot inflate its own evidence.

In [ ]:
INDUCE = '''You are a field linguist. Below are {n} numbered sentence pairs in {src} and {tgt}.
Using ONLY these examples, build a bilingual dictionary and short grammar notes for {tgt}.

Return JSON exactly in this shape:
{{"dictionary": [{{"form": "...", "source": "...", "meaning_en": "...", "examples": [1, 2]}}],
  "grammar": ["..."]}}

Rules:
- form: a word, stem or suffix of {tgt} spelled exactly as in the examples. Write suffixes with a leading hyphen, like "-kuna".
- source: the {src} word it corresponds to, lowercase.
- meaning_en: a short English meaning, one to three words.
- examples: ids of the pairs where you saw it.
- Give at most 250 entries, most frequent first.
- grammar: 5 to 12 one-sentence notes on word order, suffixes and anything else you can see.

Pairs:
{pairs}'''

def norm(s):
    s = unicodedata.normalize("NFC", s.lower())
    return re.sub(r"[^\w\s\-']", " ", s)

def tokens(s):
    return [t for t in norm(s).split() if t]

def induce(lang, n, pool, tgt_name=None):
    sub = pool.iloc[:n]
    pairs = "\n".join(f"{i+1}. {r.src} => {r.tgt}" for i, r in enumerate(sub.itertuples()))
    out = ask(INDUCE.format(n=n, src=LANGS[lang]["src_name"], tgt=tgt_name or LANGS[lang]["name"], pairs=pairs), tag=f"induce {lang} {n}")
    entries = []
    for e in out.get("dictionary", []):
        form = str(e.get("form", "")).strip(); source = str(e.get("source", "")).strip().lower()
        if not form: continue
        bare = norm(form).strip("- ")
        src_toks = set(tokens(source))
        verified = []
        for ex in e.get("examples", []) or []:
            try: i = int(ex) - 1
            except Exception: continue
            if 0 <= i < len(sub):
                row = sub.iloc[i]
                if bare and bare in norm(row.tgt) and (not src_toks or src_toks & set(tokens(row.src))):
                    verified.append(i)
        entries.append({"form": form, "source": source, "meaning_en": str(e.get("meaning_en", "")).strip().lower(),
                        "claimed": len(e.get("examples", []) or []), "evidence": len(set(verified))})
    d = pd.DataFrame(entries, columns=["form", "source", "meaning_en", "claimed", "evidence"])
    return {"dictionary": d, "grammar": out.get("grammar", []), "n": n}

LEX = {}
lex = induce("quechua", 50, DATA["quechua"][0])
print("Grammar notes the model wrote from 50 Quechua examples:")
for g in lex["grammar"]: print(" -", g)
lex["dictionary"].head(12)

## 5. Translation, and the confidence flag

The translator gets the dictionary, the grammar notes and a batch of test sentences at a time. It returns a translation, a word-by-word gloss and the source words it could not find.

The **confidence flag** uses no model at all. For each content word in the source sentence, Python looks it up in the dictionary and gives it a weight from its verified evidence (two or more supporting examples counts as full weight). The score is the average weight. Below the cutoff, Rosetta tells the user to wait for an interpreter.

In [ ]:
TRANSLATE = '''You translate from {src} into {tgt} using ONLY the dictionary and grammar notes below,
which were learned from a small set of examples. Build each translation from dictionary forms and the grammar notes.
If a word is not in the dictionary, keep the {src} word unchanged and list it under "unknown".

Grammar notes:
{grammar}

Dictionary (form = source word : English meaning):
{dictionary}

Return JSON exactly in this shape:
{{"items": [{{"id": 1, "translation": "...", "gloss": [{{"form": "...", "meaning": "..."}}], "unknown": ["..."]}}]}}

Sentences:
{sentences}'''

FEWSHOT = '''Here are example sentence pairs in {src} and {tgt}:
{pairs}

Translate each sentence below from {src} into {tgt}. Return JSON exactly in this shape:
{{"items": [{{"id": 1, "translation": "..."}}]}}

Sentences:
{sentences}'''

ZEROSHOT = '''Translate each sentence below from {src} into {tgt}. Return JSON exactly in this shape:
{{"items": [{{"id": 1, "translation": "..."}}]}}

Sentences:
{sentences}'''

STOP = set('''a al algo como con de del el ella ellas ellos en es esta este esto fue ha la las le lo los me mi muy no nos o para pero por que se si sin su sus te tu un una uno y ya yo
a an and are as at be but by for from had has have he her his i in is it its me my of on or our she so that the their them they this to was we were with you your'''.split())

def content_words(s):
    return [t for t in tokens(s) if t not in STOP and not t.isdigit()]

def confidence(src, d):
    words = content_words(src)
    if not words or d is None or d.empty: return 0.0
    ev = defaultdict(int)
    for r in d.itertuples():
        for t in tokens(r.source): ev[t] = max(ev[t], r.evidence)
    return float(np.mean([min(1.0, ev.get(w, 0) / 2) for w in words]))

def batches(df):
    for i in range(0, len(df), BATCH): yield df.iloc[i:i + BATCH]

def run_translate(lang, test, lex=None, mode="rosetta", pool=None, n=0, tgt_name=None):
    src, tgt = LANGS[lang]["src_name"], tgt_name or LANGS[lang]["name"]
    out = {}
    for b in batches(test):
        sents = "\n".join(f"{j+1}. {s}" for j, s in enumerate(b.src))
        if mode == "rosetta":
            d = lex["dictionary"]
            dic = "\n".join(f"{r.form} = {r.source} : {r.meaning_en}" for r in d.itertuples())
            p = TRANSLATE.format(src=src, tgt=tgt, grammar="\n".join("- " + g for g in lex["grammar"]), dictionary=dic, sentences=sents)
        elif mode == "fewshot":
            pairs = "\n".join(f"{r.src} => {r.tgt}" for r in pool.iloc[:n].itertuples())
            p = FEWSHOT.format(src=src, tgt=tgt, pairs=pairs, sentences=sents)
        else:
            p = ZEROSHOT.format(src=src, tgt=tgt, sentences=sents)
        res = ask(p, tag=f"{mode} {lang} {n}")
        items = {int(it.get("id", 0)): it for it in res.get("items", []) if str(it.get("id", "")).isdigit() or isinstance(it.get("id"), int)}
        for j, pid in enumerate(b.pid):
            out[pid] = items.get(j + 1, {"translation": "", "gloss": [], "unknown": []})
    return out

def score(lang, test, hyps, cond, n, lex=None):
    rows = []
    d = lex["dictionary"] if lex else None
    for r in test.itertuples():
        h = hyps[r.pid]
        hyp = str(h.get("translation", ""))
        rows.append({"lang": lang, "condition": cond, "n": n, "pid": r.pid, "source": r.src, "reference": r.tgt,
                     "output": hyp, "chrf": sacrebleu.sentence_chrf(hyp, [r.tgt]).score,
                     "confidence": confidence(r.src, d) if d is not None else np.nan,
                     "gloss": h.get("gloss", []), "unknown": h.get("unknown", [])})
    return pd.DataFrame(rows)

pool, test = DATA["quechua"]
demo = score("quechua", test.iloc[:BATCH], run_translate("quechua", test.iloc[:BATCH], lex=lex), "rosetta", 50, lex)
demo[["source", "output", "reference", "chrf", "confidence"]]

## 6. Evaluation

Every number here comes from the 50 held-out test sentences for each language. The metric is **chrF**, which compares character sequences between Rosetta's output and the professional translation. It suits Quechua and Shipibo-Konibo, which build long words from many suffixes, where whole-word matching is too harsh. Scores run from 0 to 100.

| Condition | What the model gets | Why it is here |
| --- | --- | --- |
| Zero-shot | Nothing but the instruction | How much the model already knows |
| Few-shot | N example pairs pasted in | The simple baseline Rosetta must beat |
| Rosetta | Its own dictionary and grammar from N examples | Our method |
| Rosetta, cipher | Same, but every letter of the language consistently swapped | Tests whether Rosetta learns from the examples or recalls the language from its training data |

Quechua gets the full grid. Shipibo-Konibo and Pirahã get a reduced grid to stay within free-tier rate limits.

In [ ]:
def make_cipher(texts, seed=SEED):
    letters = sorted({c for t in texts for c in t.lower() if c.isalpha()})
    shuffled = letters[:]; random.Random(seed).shuffle(shuffled)
    m = dict(zip(letters, shuffled))
    def enc(s):
        return "".join((m[c.lower()].upper() if c.isupper() else m[c]) if c.lower() in m else c for c in s)
    return enc

RESULTS, LEXICONS = [], {}
for lang, (pool, test) in DATA.items():
    sizes = SIZES_FULL if LANGS[lang]["plan"] == "full" else SIZES_REDUCED
    print(f"== {LANGS[lang]['name']}")
    RESULTS.append(score(lang, test, run_translate(lang, test, mode="zeroshot"), "zero-shot", 0))
    for n in sizes:
        RESULTS.append(score(lang, test, run_translate(lang, test, mode="fewshot", pool=pool, n=n), "few-shot", n))
        lx = induce(lang, n, pool); LEXICONS[(lang, n)] = lx
        RESULTS.append(score(lang, test, run_translate(lang, test, lex=lx), "rosetta", n, lx))
        print(f"   n={n}: dictionary entries {len(lx['dictionary'])}")
    if LANGS[lang]["plan"] == "full":
        enc = make_cipher(list(pool.tgt) + list(test.tgt))
        cpool, ctest = pool.assign(tgt=pool.tgt.map(enc)), test.assign(tgt=test.tgt.map(enc))
        for n in CIPHER_SIZES:
            lx = induce(lang, n, cpool, tgt_name="Language X")
            RESULTS.append(score(lang, ctest, run_translate(lang, ctest, lex=lx, tgt_name="Language X"), "rosetta, cipher", n, lx))

ALL = pd.concat(RESULTS, ignore_index=True)
ALL.to_csv("outputs/sentence_results.csv", index=False)
save_cache()
print("Model calls this run:", dict(CALLS))

In [ ]:
def corpus_chrf(g):
    return sacrebleu.corpus_chrf(list(g.output.astype(str)), [list(g.reference)]).score

SUMMARY = (ALL.groupby(["lang", "condition", "n"]).apply(corpus_chrf).rename("chrF").reset_index())
SUMMARY["language"] = SUMMARY.lang.map(lambda k: LANGS[k]["name"])
SUMMARY.to_csv("outputs/summary.csv", index=False)
SUMMARY.pivot_table(index=["language", "n"], columns="condition", values="chrF").round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
styles = {"few-shot": ("#8a8a8a", "o"), "rosetta": ("#1f5f8b", "s"), "rosetta, cipher": ("#c0642f", "^")}
for ax, lang in zip(axes, LANGS):
    s = SUMMARY[SUMMARY.lang == lang]
    for cond, (col, mk) in styles.items():
        t = s[s.condition == cond].sort_values("n")
        if len(t): ax.plot(t.n, t.chrF, marker=mk, color=col, label=cond)
    z = s[s.condition == "zero-shot"].chrF
    if len(z): ax.axhline(z.iloc[0], ls="--", color="#bbbbbb", label="zero-shot")
    ax.set_title(LANGS[lang]["name"]); ax.set_xlabel("example pairs given"); ax.set_xscale("log", base=2)
    ax.set_xticks([25, 50, 100, 200]); ax.set_xticklabels([25, 50, 100, 200])
axes[0].set_ylabel("chrF on 50 unseen test sentences"); axes[0].legend(frameon=False)
plt.tight_layout(); plt.savefig("outputs/learning_curves.png", dpi=150); plt.show()

### Real examples, side by side

The brief asks for real outputs next to the correct translation. For each language: Rosetta's output from 200 examples, the professional translation, and the confidence flag.

In [ ]:
best = ALL[(ALL.condition == "rosetta") & (ALL.n == 200)].copy()
best["flag"] = np.where(best.confidence >= CONFIDENCE_CUTOFF, "use", "wait for interpreter")
for lang in LANGS:
    t = best[best.lang == lang].sort_values("chrf", ascending=False)
    print(f"\n=== {LANGS[lang]['name']}: three strongest and three weakest")
    display(pd.concat([t.head(3), t.tail(3)])[["source", "output", "reference", "chrf", "confidence", "flag"]].round(2))

### Does the confidence flag work?

If the flag is useful, translations it marks "use" should score higher than ones it marks "wait for interpreter". We also report the rank correlation between confidence and sentence chrF. The cutoff (0.6) was fixed before looking at test results.

In [ ]:
from scipy.stats import spearmanr
rows = []
for lang in LANGS:
    t = best[best.lang == lang]
    rho = spearmanr(t.confidence, t.chrf).correlation if t.confidence.nunique() > 1 else np.nan
    rows.append({"language": LANGS[lang]["name"],
                 "flagged use (count)": int((t.flag == "use").sum()),
                 "chrF when flagged use": t[t.flag == "use"].chrf.mean(),
                 "chrF when flagged wait": t[t.flag != "use"].chrf.mean(),
                 "rank correlation": rho})
CONF = pd.DataFrame(rows).round(2); CONF

### Checking the dictionary against linguists (Pirahã)

The Pirahã corpus comes with word-by-word glosses written by the linguists who recorded it. We use them as an answer key that the model never sees: for each word Rosetta put in its dictionary, does the corpus gloss for that word share a word with Rosetta's English meaning?

In [ ]:
pool, test = DATA["piraha"]
gold = defaultdict(set)
for g in RAW["piraha"].gloss:
    for form, gl in g: gold[form].add(gl)
rows = []
for n in SIZES_REDUCED:
    d = LEXICONS[("piraha", n)]["dictionary"]
    stems = d[~d.form.str.startswith("-")]
    known = stems[stems.form.str.lower().isin(gold)]
    agree = [bool(set(tokens(r.meaning_en)) & set(w for gl in gold[r.form.lower()] for w in tokens(gl))) for r in known.itertuples()]
    rows.append({"examples": n, "dictionary words": len(stems), "found in corpus lexicon": len(known),
                 "meaning agrees with linguists": int(sum(agree)), "agreement rate": round(np.mean(agree), 2) if agree else np.nan})
GOLD = pd.DataFrame(rows); GOLD

### Where it fails

The five lowest-scoring Rosetta translations for each language (200 examples). Read these before trusting any average.

In [ ]:
worst = best.sort_values("chrf").groupby("lang").head(5)
worst[["lang", "source", "output", "reference", "chrf", "unknown"]]

## 7. The shared concept map

Every language hangs its words off the same 244 concepts: the Swadesh 200 and Leipzig-Jakarta 100 basic vocabulary lists, plus a few intake words such as pain, fever and medicine, all with standard IDs from [Concepticon](https://concepticon.clld.org/). A Pirahã word and a Quechua word for "water" attach to the same node, so languages that have never shared a dictionary connect through meaning.

Coverage tells a worker what the tool can handle: the share of each topic's concepts that Rosetta has a word for. When one word in a language covers two concepts (a colexification), the map draws a link between them for that language.

In [ ]:
# Build the concept list from Concepticon: Swadesh 200 + Leipzig-Jakarta 100 + a few intake concepts.
CC = "https://raw.githubusercontent.com/concepticon/concepticon-data/master/concepticondata/"
if not os.path.exists("data/concepts.csv"):
    cat = pd.read_csv(CC + "concepticon.tsv", sep="\t", dtype=str).set_index("ID")
    rows = {}
    for fname, label in [("Swadesh-1952-200", "Swadesh 1952"), ("Tadmor-2009-100", "Leipzig-Jakarta (Tadmor 2009)")]:
        for r in pd.read_csv(CC + "conceptlists/" + fname + ".tsv", sep="\t", dtype=str).itertuples():
            cid = r.CONCEPTICON_ID
            if not isinstance(cid, str): continue
            if cid in rows: rows[cid]["lists"] += "; " + label; continue
            rows[cid] = {"concepticon_id": cid, "gloss": cat.loc[cid, "GLOSS"], "english": r.ENGLISH,
                         "field": cat.loc[cid, "SEMANTICFIELD"], "lists": label}
    INTAKE = ["PAIN", "MEDICINE", "SICK", "HELP", "HOSPITAL", "WATER", "FOOD", "HOUSE", "CHILD", "HUSBAND", "WIFE",
              "MONEY", "SCHOOL", "NAME", "FAMILY", "FEVER", "TODAY", "TOMORROW", "YESTERDAY"]
    for g in INTAKE:
        hit = cat[cat.GLOSS == g]
        if len(hit) and hit.index[0] not in rows:
            cid = hit.index[0]
            rows[cid] = {"concepticon_id": cid, "gloss": g, "english": g.lower(), "field": hit.SEMANTICFIELD.iloc[0], "lists": "intake additions"}
    pd.DataFrame(rows.values()).to_csv("data/concepts.csv", index=False)
CONCEPTS = pd.read_csv("data/concepts.csv", dtype=str)
print(len(CONCEPTS), "concepts in", CONCEPTS.field.nunique(), "topics")

def concept_keys(r):
    keys = {r.gloss.lower()}
    keys |= {k.strip() for k in re.split(r"[(),;/]", r.english.lower()) if k.strip()}
    keys |= {k[3:] for k in list(keys) if k.startswith("to ")}
    return {k for k in keys if len(k) > 1}

KEY2C = defaultdict(set)
for r in CONCEPTS.itertuples():
    for k in concept_keys(r): KEY2C[k].add(r.concepticon_id)

def match_concepts(meaning):
    m = norm(meaning).strip()
    m = re.sub(r"^(to|a|an|the)\s+", "", m)
    hits = set(KEY2C.get(m, set()))
    if not hits:
        for t in m.split():
            hits |= KEY2C.get(t, set())
    return hits

WORDS = []
for lang in LANGS:
    d = LEXICONS[(lang, 200)]["dictionary"]
    for r in d.itertuples():
        if r.form.startswith("-"): continue
        for cid in match_concepts(r.meaning_en):
            WORDS.append({"lang": lang, "form": r.form, "meaning": r.meaning_en, "evidence": int(r.evidence), "cid": cid})
WORDS = pd.DataFrame(WORDS).drop_duplicates(["lang", "form", "cid"])

field_of = dict(zip(CONCEPTS.concepticon_id, CONCEPTS.field))
cov = []
for lang in LANGS:
    known = set(WORDS[WORDS.lang == lang].cid)
    for f, g in CONCEPTS.groupby("field"):
        cov.append({"language": LANGS[lang]["name"], "topic": f, "concepts": len(g), "covered": len(known & set(g.concepticon_id))})
COV = pd.DataFrame(cov); COV["percent"] = (100 * COV.covered / COV.concepts).round(0)
COV.pivot_table(index="topic", columns="language", values="percent").fillna(0).astype(int)

In [ ]:
COLEX = []
for (lang, form), g in WORDS.groupby(["lang", "form"]):
    ids = sorted(set(g.cid))
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            COLEX.append({"lang": lang, "form": form, "a": ids[i], "b": ids[j]})
COLEX = pd.DataFrame(COLEX, columns=["lang", "form", "a", "b"])
print("Colexifications found:", len(COLEX))

G = nx.Graph()
gloss_of = dict(zip(CONCEPTS.concepticon_id, CONCEPTS.gloss))
for r in CONCEPTS.itertuples(): G.add_node("c" + r.concepticon_id, kind="concept")
for r in WORDS.itertuples(): G.add_edge("c" + r.cid, f"{r.lang}:{r.form}")
colors = {"quechua": "#1f5f8b", "shipibo": "#2e7d4f", "piraha": "#c0642f"}
pos = nx.spring_layout(G, seed=SEED, k=0.35)
plt.figure(figsize=(11, 8))
lit = {n for n in G if n.startswith("c") and G.degree(n) > 0}
nx.draw_networkx_edges(G, pos, alpha=0.15)
nx.draw_networkx_nodes(G, pos, nodelist=[n for n in G if n.startswith("c") and n not in lit], node_size=12, node_color="#dddddd")
nx.draw_networkx_nodes(G, pos, nodelist=list(lit), node_size=40, node_color="#333333")
for lang, col in colors.items():
    nx.draw_networkx_nodes(G, pos, nodelist=[n for n in G if n.startswith(lang + ":")], node_size=18, node_color=col, label=LANGS[lang]["name"])
nx.draw_networkx_labels(G, pos, labels={n: gloss_of[n[1:]].lower() for n in lit if G.degree(n) >= 2}, font_size=7)
plt.legend(frameon=False); plt.axis("off"); plt.title("Concepts (gray) lit by words each language learned from 200 examples")
plt.savefig("outputs/concept_map_2d.png", dpi=150, bbox_inches="tight"); plt.show()

### Export for the 3D map

The 3D map on the project site reads one file, `graph.json`, written here. It holds the concepts, the words each language learned, the colexification links, coverage by topic, the results table and real examples.

In [ ]:
nodes, links = [], []
for f in sorted(CONCEPTS.field.unique()):
    nodes.append({"id": "f:" + f, "kind": "field", "label": f})
for r in CONCEPTS.itertuples():
    nodes.append({"id": "c:" + r.concepticon_id, "kind": "concept", "label": r.gloss.lower(), "field": r.field, "cid": r.concepticon_id})
    links.append({"source": "f:" + r.field, "target": "c:" + r.concepticon_id, "kind": "field"})
for (lang, form), g in WORDS.groupby(["lang", "form"]):
    wid = f"w:{lang}:{form}"
    nodes.append({"id": wid, "kind": "word", "lang": lang, "label": form, "meaning": g.meaning.iloc[0], "evidence": int(g.evidence.max())})
    for cid in set(g.cid):
        links.append({"source": wid, "target": "c:" + cid, "kind": "means"})
for r in COLEX.itertuples():
    links.append({"source": "c:" + r.a, "target": "c:" + r.b, "kind": "colex", "lang": r.lang, "form": r.form})

ex = []
for lang in LANGS:
    t = best[best.lang == lang].sort_values("chrf", ascending=False)
    for r in pd.concat([t.head(3), t.tail(2)]).itertuples():
        ex.append({"lang": lang, "source": r.source, "output": r.output, "reference": r.reference,
                   "chrf": round(r.chrf, 1), "confidence": round(r.confidence, 2), "flag": r.flag})

GRAPH = {"languages": {k: {"name": v["name"], "region": v["region"], "src": v["src_name"]} for k, v in LANGS.items()},
         "nodes": nodes, "links": links,
         "coverage": COV.to_dict("records"),
         "summary": SUMMARY.round(1).to_dict("records"),
         "examples": ex, "model": MODEL}
json.dump(GRAPH, open("outputs/graph.json", "w"), ensure_ascii=False)
print(len(nodes), "nodes,", len(links), "links -> outputs/graph.json")

## 8. Try it

Type a sentence in the source language, pick a language, and Rosetta translates it with the dictionary it learned from 200 examples. Sentences that are not in the cache need an API key.

In [ ]:
def rosetta(sentence, lang="quechua"):
    lx = LEXICONS[(lang, 200)]
    t = pd.DataFrame({"src": [sentence], "tgt": [""], "pid": [0]})
    h = run_translate(lang, t, lex=lx)[0]
    c = confidence(sentence, lx["dictionary"])
    print(f"{LANGS[lang]['src_name']}: {sentence}")
    print(f"{LANGS[lang]['name']}: {h.get('translation', '')}")
    print("Gloss:", "  ".join(f"{g.get('form','')}={g.get('meaning','')}" for g in h.get("gloss", [])))
    if h.get("unknown"): print("Not in dictionary:", ", ".join(h["unknown"]))
    print(f"Confidence {c:.2f}:", "use with care" if c >= CONFIDENCE_CUTOFF else "wait for a human interpreter")

rosetta("¿Dónde le duele?", "quechua")
rosetta("The child is sick.", "piraha")

## 9. Save the outputs

This cell bundles the cache, results and `graph.json` into one zip. Upload its contents to the GitHub repo so the notebook replays without a key and the 3D map shows these results.

In [ ]:
import shutil
shutil.copy(CACHE_PATH, "outputs/model_cache.json")
shutil.make_archive("rosetta_outputs", "zip", ".", "outputs")
try:
    from google.colab import files
    files.download("rosetta_outputs.zip")
except Exception:
    print("Saved rosetta_outputs.zip")

## 10. Second run: the dictionary and the examples together

Run sections 1 to 3 first (setup, model client, data), then this cell. It gives the translator the 200 examples as well as the dictionary from the first run (`data/dictionaries_200.json`), runs every language on one model, and reruns few-shot on that same model so the comparison is fair. It makes about 12 model calls.

In [9]:
# ---- Rosetta v2: the dictionary AND the examples, on one model for all three languages ----
# Run sections 1 to 3 first (setup, model client, data). This cell does not need any other cell.
# It makes about 12 model calls in total, well under the free tier's 20 per model per day.

V2_MODELS = ["gemini-3.6-flash", "gemini-3.7-flash", "gemini-3-flash-preview", "gemini-flash-latest"]
CUTOFF = 0.6

def _norm(s):
    s = unicodedata.normalize("NFC", str(s).lower())
    return re.sub(r"[^\w\s\-']", " ", s)
def _toks(s): return [t for t in _norm(s).split() if t]
_STOP = set('''a al algo como con de del el ella ellas ellos en es esta este esto fue ha la las le lo los me mi muy no nos o para pero por que se si sin su sus te tu un una uno y ya yo
a an and are as at be but by for from had has have he her his i in is it its me my of on or our she so that the their them they this to was we were with you your'''.split())
def _content(s): return [t for t in _toks(s) if t not in _STOP and not t.isdigit()]

def _ask2(model, prompt, tag):
    k = hashlib.sha256((model + "\n" + prompt).encode()).hexdigest()
    if k in CACHE["responses"]:
        return json.loads(CACHE["responses"][k])
    json_fails = 0
    for attempt in range(6):
        try:
            print(f"   calling {model}: {tag} at {time.strftime('%H:%M:%S')}")
            r = client.models.generate_content(model=model, contents=prompt,
                config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json", max_output_tokens=32768))
            text = re.sub(r"^```(?:json)?\s*|\s*```$", "", (r.text or "").strip())
            data = json.loads(text)
            CACHE["responses"][k] = text; save_cache()
            return data
        except json.JSONDecodeError as e:
            json_fails += 1
            if json_fails >= 2: raise RuntimeError(f"Bad JSON on {tag}: {e}")
        except Exception as e:
            msg = str(e)
            if "PerDay" in msg or "per day" in msg.lower():
                raise RuntimeError("Daily quota reached for " + model) from e
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg or "503" in msg or "UNAVAILABLE" in msg:
                time.sleep(20 * (attempt + 1)); continue
            raise
    raise RuntimeError(f"Gave up on {tag}")

def _pick_v2():
    for m in V2_MODELS:
        for _ in range(3):
            try:
                client.models.generate_content(model=m, contents="Say OK"); return m
            except Exception as e:
                msg = str(e); print("  ", m, "->", msg[:80])
                if "503" in msg or "UNAVAILABLE" in msg: time.sleep(30); continue
                break
    raise RuntimeError("No model available right now. Try again in an hour.")

# 1. The dictionaries Rosetta learned from 200 examples in the first run, re-checked here by Python
_dpath = "data/dictionaries_200.json"
if not os.path.exists(_dpath): _download(REPO_RAW + _dpath, _dpath)
RAWDICT = json.load(open(_dpath, encoding="utf-8"))
LEX2 = {}
for lang in LANGS:
    sub = DATA[lang][0].iloc[:200]
    rows = []
    for e in RAWDICT[lang]["dictionary"]:
        form = str(e.get("form", "")).strip(); source = str(e.get("source", "")).strip().lower()
        if not form: continue
        bare = _norm(form).strip("- "); st = set(_toks(source)); ok = set()
        for ex in e.get("examples") or []:
            try: i = int(ex) - 1
            except Exception: continue
            if 0 <= i < len(sub):
                r = sub.iloc[i]
                if bare and bare in _norm(r.tgt) and (not st or st & set(_toks(r.src))): ok.add(i)
        rows.append({"form": form, "source": source, "meaning_en": str(e.get("meaning_en", "")).lower(), "evidence": len(ok)})
    LEX2[lang] = {"dictionary": pd.DataFrame(rows), "grammar": RAWDICT[lang].get("grammar", [])}
    print(f"{LANGS[lang]['name']}: {len(rows)} dictionary entries, {sum(r['evidence'] > 0 for r in rows)} with checked evidence")

# 2. Confidence, still computed by Python only: a source word counts as known if the checked
#    dictionary has it, or if it appears in at least two of the 200 example sentences.
def confidence2(src, lang):
    words = _content(src)
    if not words: return 0.0
    d = LEX2[lang]["dictionary"]; ev = defaultdict(int)
    for r in d.itertuples():
        for t in _toks(r.source): ev[t] = max(ev[t], r.evidence)
    seen = Counter(t for s in DATA[lang][0].iloc[:200].src for t in set(_toks(s)))
    return float(np.mean([min(1.0, max(ev.get(w, 0), seen.get(w, 0)) / 2) for w in words]))

FEWSHOT2 = '''Here are example sentence pairs in {src} and {tgt}:
{pairs}

Translate each sentence below from {src} into {tgt}. Return JSON exactly in this shape:
{{"items": [{{"id": 1, "translation": "..."}}]}}

Sentences:
{sentences}'''

HYBRID = '''You translate from {src} into {tgt}. You have {n} example sentence pairs and a dictionary
with grammar notes that were learned from those same examples. Use both.
Translate every word. If you are not sure of a word, give your best guess and list the {src} word under "unsure".
Keep a {src} word unchanged only if it is a name or a loanword that the examples also keep.

Grammar notes:
{grammar}

Dictionary (form = {src} word : English meaning):
{dictionary}

Examples:
{pairs}

Return JSON exactly in this shape:
{{"items": [{{"id": 1, "translation": "...", "gloss": [{{"form": "...", "meaning": "..."}}], "unsure": ["..."]}}]}}

Sentences:
{sentences}'''

def _translate(model, lang, mode):
    pool, test = DATA[lang]
    src, tgt = LANGS[lang]["src_name"], LANGS[lang]["name"]
    pairs = "\n".join(f"{r.src} => {r.tgt}" for r in pool.iloc[:200].itertuples())
    d = LEX2[lang]["dictionary"]
    dic = "\n".join(f"{r.form} = {r.source} : {r.meaning_en}" for r in d.itertuples())
    out = {}
    def do(b):
        sents = "\n".join(f"{j+1}. {s}" for j, s in enumerate(b.src))
        if mode == "hybrid":
            p = HYBRID.format(src=src, tgt=tgt, n=200, grammar="\n".join("- " + g for g in LEX2[lang]["grammar"]), dictionary=dic, pairs=pairs, sentences=sents)
        else:
            p = FEWSHOT2.format(src=src, tgt=tgt, pairs=pairs, sentences=sents)
        try:
            res = _ask2(model, p, f"{mode} {lang} ({len(b)} sentences)")
        except RuntimeError as e:
            if "Bad JSON" in str(e) and len(b) > 1:
                h = len(b) // 2; do(b.iloc[:h]); do(b.iloc[h:]); return
            raise
        items = {int(it["id"]): it for it in res.get("items", []) if str(it.get("id", "")).isdigit() or isinstance(it.get("id"), int)}
        for j, pid in enumerate(b.pid):
            out[pid] = items.get(j + 1, {"translation": ""})
    for i in range(0, len(test), 25): do(test.iloc[i:i + 25])
    return out

# 3. Run: few-shot and Rosetta + examples, same model, same 200 examples, same 50 test sentences
V2_MODEL = CACHE["meta"].get("v2_model") or _pick_v2()
CACHE["meta"]["v2_model"] = V2_MODEL; save_cache()
print("Model for every language:", V2_MODEL)
rows2 = []
for lang in LANGS:
    pool, test = DATA[lang]
    print("==", LANGS[lang]["name"])
    for mode, cond in [("fewshot", "few-shot"), ("hybrid", "rosetta + examples")]:
        hyps = _translate(V2_MODEL, lang, mode)
        for r in test.itertuples():
            h = hyps[r.pid]; hyp = str(h.get("translation", ""))
            c = confidence2(r.src, lang)
            rows2.append({"lang": lang, "condition": cond, "n": 200, "pid": r.pid, "source": r.src, "reference": r.tgt, "output": hyp,
                          "chrf": sacrebleu.sentence_chrf(hyp, [r.tgt]).score, "confidence": c,
                          "flag": "use" if c >= CUTOFF else "wait for interpreter",
                          "gloss": h.get("gloss", []), "unsure": h.get("unsure", [])})
V2 = pd.DataFrame(rows2)
V2.to_csv("outputs/sentence_results_v2.csv", index=False)

# 4. Scores
S2 = (V2.groupby(["lang", "condition"]).apply(lambda g: sacrebleu.corpus_chrf(list(g.output.astype(str)), [list(g.reference)]).score)
        .rename("chrF").reset_index())
S2["language"] = S2.lang.map(lambda k: LANGS[k]["name"])
S2.to_csv("outputs/summary_v2.csv", index=False)
print("\nchrF on 50 unseen test sentences, 200 examples, all on", V2_MODEL)
display(S2.pivot_table(index="language", columns="condition", values="chrF").round(1))

FLAG2 = (V2[V2.condition == "rosetta + examples"].groupby(["lang", "flag"])
         .agg(sentences=("chrf", "size"), mean_chrF=("chrf", "mean")).round(1))
print("\nConfidence flag on Rosetta + examples (sentence-level chrF):")
display(FLAG2)
print("All languages:", V2[V2.condition == "rosetta + examples"].groupby("flag").chrf.agg(["size", "mean"]).round(1).to_dict())

# 5. Update the site data: the few-shot column becomes the same-model run, and a new column is added
_gpath = "outputs/graph.json"
if not os.path.exists(_gpath): _download(REPO_RAW + "docs/graph.json", _gpath)
GRAPH2 = json.load(open(_gpath, encoding="utf-8"))
keep = [r for r in GRAPH2["summary"] if not (r["condition"] in ("few-shot", "rosetta + examples") and r["n"] == 200)]
for r in S2.itertuples():
    keep.append({"lang": r.lang, "condition": r.condition, "n": 200, "chrF": round(r.chrF, 1), "language": r.language})
GRAPH2["summary"] = keep
GRAPH2["v2_model"] = V2_MODEL
json.dump(GRAPH2, open("outputs/graph.json", "w"), ensure_ascii=False)
print("\nUpdated outputs/graph.json. Download it with the zip cell, or from the Files panel.")


Southern Quechua: 65 dictionary entries, 41 with checked evidence
Shipibo-Konibo: 15 dictionary entries, 11 with checked evidence
Pirahã: 111 dictionary entries, 107 with checked evidence
Model for every language: gemini-3.6-flash
== Southern Quechua
   calling gemini-3.6-flash: fewshot quechua (25 sentences) at 01:17:46
   calling gemini-3.6-flash: fewshot quechua (25 sentences) at 01:18:09
   calling gemini-3.6-flash: hybrid quechua (25 sentences) at 01:18:48
   calling gemini-3.6-flash: hybrid quechua (25 sentences) at 01:19:39
== Shipibo-Konibo
   calling gemini-3.6-flash: fewshot shipibo (25 sentences) at 01:20:19
   calling gemini-3.6-flash: fewshot shipibo (25 sentences) at 01:20:48
   calling gemini-3.6-flash: hybrid shipibo (25 sentences) at 01:21:05
   calling gemini-3.6-flash: hybrid shipibo (25 sentences) at 01:21:36
== Pirahã
   calling gemini-3.6-flash: fewshot piraha (25 sentences) at 01:22:10
   calling gemini-3.6-flash: fewshot piraha (25 sentences) at 01:22:41
   call

/tmp/ipykernel_725/4195203741.py:161: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  S2 = (V2.groupby(["lang", "condition"]).apply(lambda g: sacrebleu.corpus_chrf(list(g.output.astype(str)), [list(g.reference)]).score)


condition,few-shot,rosetta + examples
language,,
Pirahã,42.2,42.2
Shipibo-Konibo,44.8,44.5
Southern Quechua,51.3,48.2



Confidence flag on Rosetta + examples (sentence-level chrF):


sentences  mean_chrF
lang    flag                                      
piraha  use                          42       40.2
        wait for interpreter          8       36.1
quechua use                          10       53.1
        wait for interpreter         40       48.0
shipibo use                          10       49.6
        wait for interpreter         40       43.1

All languages: {'size': {'use': 62, 'wait for interpreter': 88}, 'mean': {'use': 43.8, 'wait for interpreter': 44.7}}

Updated outputs/graph.json. Download it with the zip cell, or from the Files panel.


## Limits

* One test set of 50 short sentences per language. Differences of a few chrF points are within noise.
* No speaker of these languages checked the outputs. For Pirahã, the linguists' glosses stand in as a partial check of the dictionary, not of the translations.
* The Quechua and Shipibo-Konibo sentences are translations of Spanish news-style text, not intake questions.
* The Pirahã corpus has been public since 2016, so the model may have seen it. The cipher test on Quechua is how we measure that kind of prior knowledge.

## Credits

* Data: [AmericasNLP 2021](https://github.com/AmericasNLP/americasnlp2021) (Mager et al. 2021); [MIT Pirahã Corpus](https://osf.io/kt2e8/) (Futrell, Stearns, Everett, Piantadosi and Gibson 2016, CC BY-SA); [Concepticon](https://concepticon.clld.org/) concept lists (Swadesh 1952; Tadmor 2009).
* Model: Google Gemini API, free tier.
* Libraries: pandas, sacrebleu, networkx, matplotlib, scipy, google-genai.
* AI assistance: Claude (Anthropic) helped write and debug the code and draft the documentation.